# LEMON Full Batch Preprocessing Only

This notebook is a clean, batch-only version of preprocessing.
It avoids the single-subject demo flow and runs directly on all subjects.

## Sub-steps
1. Imports and environment checks
2. Paths and parameters
3. Helper functions (bad channels, bad segments)
4. Subject-level pipeline function
5. Batch loop (all subjects)
6. Summary and CSV export

In [1]:
# Sub-step 1: Imports and environment checks
import os
import glob
import time
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from mne_icalabel import label_components

wandb = None
WANDB_AVAILABLE = False
try:
    import wandb as wandb_module
    wandb = wandb_module
    WANDB_AVAILABLE = True
except ImportError:
    pass

def detect_cuda_root():
    candidate_envs = [os.getenv('CUDA_PATH'), os.getenv('CUDA_HOME')]
    candidate_envs.extend([os.getenv('CUDA_PATH_V13_0'), os.getenv('CUDA_PATH_V12_9'), os.getenv('CUDA_PATH_V12_8')])
    candidate_dirs = []
    for candidate in candidate_envs:
        if candidate:
            candidate_dirs.append(Path(candidate))
    candidate_dirs.extend([
        Path(r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0'),
        Path(r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9'),
        Path(r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.8'),
        Path(r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.6'),
        Path(r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.5'),
        Path(r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4'),
    ])
    for candidate in candidate_dirs:
        if candidate and candidate.exists():
            return str(candidate)
    return None

CUDA_ROOT = detect_cuda_root()
if CUDA_ROOT:
    os.environ['CUDA_PATH'] = CUDA_ROOT
    os.environ['CUDA_HOME'] = CUDA_ROOT
    print(f'CUDA toolkit root detected: {CUDA_ROOT}')
else:
    print('CUDA toolkit root not found. CuPy may still run, but the CUDA_PATH warning can remain.')

CUPY_AVAILABLE = False
try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    cp = None

print(f'MNE version: {mne.__version__}')
mne.set_log_level('WARNING')
RANDOM_STATE = 42

MNE_CUDA = False
try:
    if CUPY_AVAILABLE:
        mne.utils.set_config('MNE_USE_CUDA', 'true', set_env=True)
        try:
            mne.cuda.init_cuda(verbose=True)
            MNE_CUDA = True
            print('CUDA mode enabled: CuPy is available and MNE CUDA initialized.')
        except Exception as exc:
            print(f'CUDA initialization failed ({type(exc).__name__}: {exc}). Falling back to CPU.')
            mne.utils.set_config('MNE_USE_CUDA', 'false', set_env=True)
    else:
        mne.utils.set_config('MNE_USE_CUDA', 'false', set_env=True)
        print('CuPy is not installed. Falling back to CPU.')
except Exception as exc:
    print(f'Could not configure CUDA ({type(exc).__name__}: {exc}). Falling back to CPU.')
    try:
        mne.utils.set_config('MNE_USE_CUDA', 'false', set_env=True)
    except Exception:
        pass
    MNE_CUDA = False

print(f'Forcing compute mode: {"CUDA" if MNE_CUDA else "CPU"}')

CUDA toolkit root detected: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.2
MNE version: 1.11.0
Now using CUDA device 0
Enabling CUDA with 3.22 GiB available memory
CUDA mode enabled: CuPy is available and MNE CUDA initialized.
Forcing compute mode: CUDA


In [2]:
#parameters
LEMON_DIR = r'G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\EEG_MPILMBB_LEMON\\EEG_Raw_BIDS_ID'
OUTPUT_DIR = r'G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\data\\LEMON_preprocessed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SFREQ = 250
LINE_NOISE_FREQ = 50
HIGHPASS_FREQ = 1.0
LOWPASS_FREQ = 100.0
IC_REJECTION_THRESHOLD = 0.80
EPOCH_DURATION = 4.0
EPOCH_OVERLAP = 0.5
AMPLITUDE_REJECT_UV = 250e-6

# QC thresholds (configurable)
IC_REJECT_FRACTION_FLAG = 0.25  # flag if >=25% of ICA components rejected
EPOCH_KEEP_RATIO_FLAG = 0.5     # flag if kept epochs <=50% of before

subject_dirs = sorted(glob.glob(os.path.join(LEMON_DIR, 'sub-*')))
print(f'Found {len(subject_dirs)} subjects')
print(f'Output dir: {OUTPUT_DIR}')

Found 220 subjects
Output dir: G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\data\\LEMON_preprocessed


In [3]:
# Sub-step 2b: Weights & Biases tracking setup
WANDB_ENTITY = 'amahbe223043-united-international-university'
WANDB_PROJECT = 'LEMON_DATASET_preprocessing'
WANDB_RUN_NAME = f'lemon-preprocessing-{time.strftime("%Y%m%d-%H%M%S")}'

# Increase init timeout to avoid startup failures on slow/unreliable networks.
WANDB_INIT_TIMEOUT = int(os.getenv('WANDB_INIT_TIMEOUT', '180'))

wandb_run = None
wandb_table = None
wandb_subject_rows = []

if WANDB_AVAILABLE:
    try:
        wandb_api_key = os.getenv('WANDB_API_KEY')

        # If key is present in env, use it explicitly; otherwise rely on existing `wandb login` session.
        if wandb_api_key:
            wandb.login(key=wandb_api_key)

        wandb_run = wandb.init(
            entity=WANDB_ENTITY,
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config={
                'target_sfreq': TARGET_SFREQ,
                'line_noise_freq': LINE_NOISE_FREQ,
                'highpass_freq': HIGHPASS_FREQ,
                'lowpass_freq': LOWPASS_FREQ,
                'ic_rejection_threshold': IC_REJECTION_THRESHOLD,
                'epoch_duration': EPOCH_DURATION,
                'epoch_overlap': EPOCH_OVERLAP,
                'amplitude_reject_uv': AMPLITUDE_REJECT_UV,
                'random_state': RANDOM_STATE,
                'use_cuda': MNE_CUDA,
                'total_subjects': len(subject_dirs),
            },
            reinit='finish_previous',
            settings=wandb.Settings(init_timeout=WANDB_INIT_TIMEOUT),
        )
        wandb.define_metric('preprocessing/subject_index')
        wandb.define_metric('preprocessing/*', step_metric='preprocessing/subject_index')
        print(f'W&B tracking enabled: {WANDB_ENTITY}/{WANDB_PROJECT}')
        print(f'W&B run URL: {wandb_run.get_url()}')
    except Exception as exc:
        print(f'W&B setup failed ({type(exc).__name__}: {exc}). Retrying in offline mode...')
        try:
            wandb_run = wandb.init(
                entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                name=WANDB_RUN_NAME,
                mode='offline',
                config={
                    'target_sfreq': TARGET_SFREQ,
                    'line_noise_freq': LINE_NOISE_FREQ,
                    'highpass_freq': HIGHPASS_FREQ,
                    'lowpass_freq': LOWPASS_FREQ,
                    'ic_rejection_threshold': IC_REJECTION_THRESHOLD,
                    'epoch_duration': EPOCH_DURATION,
                    'epoch_overlap': EPOCH_OVERLAP,
                    'amplitude_reject_uv': AMPLITUDE_REJECT_UV,
                    'random_state': RANDOM_STATE,
                    'use_cuda': MNE_CUDA,
                    'total_subjects': len(subject_dirs),
                },
                settings=wandb.Settings(init_timeout=WANDB_INIT_TIMEOUT),
            )
            wandb.define_metric('preprocessing/subject_index')
            wandb.define_metric('preprocessing/*', step_metric='preprocessing/subject_index')
            print('W&B offline mode enabled. Run `wandb sync` later to upload logs.')
        except Exception as offline_exc:
            print(f'W&B offline fallback failed ({type(offline_exc).__name__}: {offline_exc}). Continuing without W&B logging.')
            wandb_run = None
else:
    print('W&B is not installed. Add wandb to requirements.txt or install it to enable tracking.')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\milon\_netrc.


wandb: Currently logged in as: amahbe223043 (amahbe223043-united-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


W&B tracking enabled: amahbe223043-united-international-university/LEMON_DATASET_preprocessing
W&B run URL: https://wandb.ai/amahbe223043-united-international-university/LEMON_DATASET_preprocessing/runs/tlja6zwb


In [4]:
#bad-channel helper (PyPREP + RANSAC fallback)
def detect_bad_channels(raw, random_state=42, include_ransac=True):
    try:
        from pyprep.find_noisy_channels import NoisyChannels
    except ImportError as exc:
        raise ImportError('PyPREP is required. Install with: pip install pyprep') from exc

    picks_eeg = mne.pick_types(raw.info, eeg=True, eog=False, exclude=[])
    eeg_names_all = [raw.ch_names[p] for p in picks_eeg]
    eeg_names = [ch for ch in eeg_names_all if ch != 'FCz']
    raw_prep = raw.copy().pick_channels(eeg_names, ordered=True)

    noisy = NoisyChannels(raw_prep, random_state=random_state)
    noisy.find_bad_by_nan_flat()
    noisy.find_bad_by_deviation()
    noisy.find_bad_by_hfnoise()
    noisy.find_bad_by_correlation()

    ransac_used = False
    if include_ransac:
        try:
            noisy.find_bad_by_ransac()
            ransac_used = True
        except Exception as exc:
            warnings.warn(
                f'PyPREP RANSAC failed ({type(exc).__name__}: {exc}). Continuing without RANSAC.',
                RuntimeWarning
            )

    bad_channels = sorted(set(noisy.get_bads()))
    return bad_channels, ransac_used

bad time segment helper

In [ ]:

def annotate_bad_segments(raw, threshold_uv=250, window_s=0.5, use_cuda=False):
    sfreq = raw.info['sfreq']
    window_samples = int(window_s * sfreq)
    picks_eeg = mne.pick_types(raw.info, eeg=True, eog=False)#pick all EEG channels except EOG
    data = raw.get_data(picks=picks_eeg) * 1e6
    n_windows = data.shape[1] // window_samples
    bad_annotations = []
    annotation_orig_time = raw.annotations.orig_time

    use_gpu = bool(use_cuda and CUPY_AVAILABLE and cp is not None)
    xp = cp if use_gpu else np

    if use_gpu:
        data = cp.asarray(data)

    for w in range(n_windows):
        start = w * window_samples
        end = start + window_samples
        window_data = data[:, start:end]
        ptp = xp.ptp(window_data, axis=1)  #peak to peak amplitude for each channel in the window
        if use_gpu:
            artifact_found = bool(cp.any(ptp > threshold_uv).item())
        else:
            artifact_found = bool(np.any(ptp > threshold_uv))
        if artifact_found:
            bad_annotations.append((start / sfreq, window_s, 'BAD_artifact'))

    if bad_annotations:
        onsets = [a[0] for a in bad_annotations]  
        durations = [a[1] for a in bad_annotations]  # extract durations from the annotations list
        desc = [a[2] for a in bad_annotations]  # extract descriptions from the annotations list
        bad_annot = mne.Annotations(onset=onsets, duration=durations, description=desc, orig_time=annotation_orig_time)
        raw.set_annotations(raw.annotations + bad_annot)  # add the new annotations to the raw object

    return len(bad_annotations) * window_s #return the total duration of bad segments in seconds

subject level preprocessing pipeline

In [6]:

def preprocess_lemon_subject(vhdr_file, output_dir,#vhdr is the brainvision header file path
                           target_sfreq=250,
                           line_noise_freq=50,
                           highpass_freq=1.0,
                           lowpass_freq=100.0,
                           ic_threshold=0.80,
                           epoch_duration=4.0,
                           epoch_overlap_frac=0.5,
                           amplitude_reject_uv=250e-6,
                           random_state=42,
                           use_cuda=False):
    n_jobs = 'cuda' if use_cuda else 1
    subject_id = os.path.basename(os.path.dirname(os.path.dirname(vhdr_file)))
    result = {
        'subject': subject_id,
        'status': 'success',
        'n_bad_channels': 0,
        'bad_channel_names': [],
        'n_rejected_ics': 0,
        'reject_ics_fraction': np.nan,
        'bad_segments_s': 0.0,
        'n_epochs_before': 0,
        'n_epochs_after': 0,
        'epoch_keep_ratio': np.nan,
    }

    try:
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=True, verbose=False)

        # Channel setup
        raw.set_channel_types({'VEOG': 'eog'})
        if 'FCz' not in raw.ch_names:
            raw = mne.add_reference_channels(raw, ref_channels=['FCz'])
        raw.set_montage(mne.channels.make_standard_montage('standard_1005'), on_missing='warn', verbose=False)

        #Filtering + resampling
        raw.notch_filter(freqs=[line_noise_freq, line_noise_freq * 2], method='spectrum_fit', verbose=False)
        #this removes powerline noise and its first harmonic,which are common contaminans in EEG data..
        raw.filter(l_freq=highpass_freq, h_freq=lowpass_freq, n_jobs=n_jobs, verbose=False)
        #this removes slow drifts and DC offset, which can interfere with ICA and other analyses 
        raw.resample(sfreq=target_sfreq, npad='auto', n_jobs=n_jobs, verbose=False)
        #we resample to 250hz to balance temporal resolution and computational efficiency and to reduce computational time 
        #also our other dataset is at 250hz so this makes it easier to work with both dataset together

        # Bad channels
        bad_chs, ransac_used = detect_bad_channels(raw, random_state=random_state, include_ransac=True)
        raw.info['bads'] = bad_chs
        result['n_bad_channels'] = len(bad_chs)
        result['bad_channel_names'] = bad_chs
        result['ransac_used'] = ransac_used

        dropped = list(raw.info['bads'])
        raw_clean = raw.copy().drop_channels(dropped) if dropped else raw.copy()

        # Re-reference
        raw_clean.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

        # ICA + ICLabel
        n_eeg = len(mne.pick_types(raw_clean.info, eeg=True, eog=False))
        if n_eeg < 2:
            raise RuntimeError(f'Not enough EEG channels for ICA after cleaning: {n_eeg}')
        n_components = n_eeg - 1
        ica = mne.preprocessing.ICA(
            n_components=n_components,
            method='infomax',
            fit_params=dict(extended=True),
            random_state=random_state,
            max_iter='auto',
            verbose=False
        )
        ica.fit(raw_clean, picks='eeg', verbose=False)

        ic_labels = label_components(raw_clean, ica, method='iclabel')
        labels = ic_labels.get('labels', [])
        probs = np.asarray(ic_labels.get('y_pred_proba', []))
        reject_label_set = {
            'eye blink',
            'muscle artifact',
            'heart beat',
            'line noise',
            'channel noise',
            'other',
        }
        reject_ics = []
        for i, label in enumerate(labels):
            prob_i = float(probs[i]) if probs.size > i else 1.0
            if label in reject_label_set and prob_i >= ic_threshold:
                reject_ics.append(i)
        ica.exclude = reject_ics
        raw_clean = ica.apply(raw_clean, verbose=False)
        result['n_rejected_ics'] = len(reject_ics)
        #fraction of rejected ICA components
        try:
            ic_frac = len(reject_ics) / max(1, n_components)
        except Exception:
            ic_frac = 1.0 if len(reject_ics) > 0 else 0.0
        result['reject_ics_fraction'] = ic_frac
        result['flag_high_ic_reject'] = bool(ic_frac >= IC_REJECT_FRACTION_FLAG)
        if result['flag_high_ic_reject']:
            warnings.warn(f"High IC rejection fraction: {ic_frac:.1%}", RuntimeWarning)

        #interpolate dropped channels
        if dropped:
            for ch in dropped:
                if ch not in raw_clean.ch_names:
                    raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])
                    # Re-apply montage so the newly added channel has a location (silences warning)
                    raw_clean.set_montage(mne.channels.make_standard_montage('standard_1005'), on_missing='warn', verbose=False)
            raw_clean.info['bads'] = dropped
            raw_clean.set_montage(mne.channels.make_standard_montage('standard_1005'), on_missing='warn', verbose=False)
            raw_clean.interpolate_bads(reset_bads=True, verbose=False)

        # Bad segments + epoching
        bad_s = annotate_bad_segments(raw_clean, threshold_uv=250, window_s=0.5, use_cuda=use_cuda)
        result['bad_segments_s'] = bad_s

        raw_eeg = raw_clean.copy().pick('eeg')
        epochs = mne.make_fixed_length_epochs(
            raw_eeg,
            duration=epoch_duration,
            overlap=epoch_duration * epoch_overlap_frac,
            preload=True,
            verbose=False
        )

        n_before = len(epochs)
        epochs.drop_bad(reject=dict(eeg=amplitude_reject_uv), verbose=False)
        result['n_epochs_before'] = n_before
        result['n_epochs_after'] = len(epochs)
        result['epoch_shape'] = epochs.get_data().shape
        # QC: epoch keep ratio
        try:
            keep_ratio = result['n_epochs_after'] / max(1, result['n_epochs_before'])
        except Exception:
            keep_ratio = 0.0
        result['epoch_keep_ratio'] = keep_ratio
        result['flag_epoch_loss'] = bool(keep_ratio <= EPOCH_KEEP_RATIO_FLAG)
        if result['flag_epoch_loss']:
            warnings.warn(f"Large epoch loss: kept {keep_ratio:.1%} of epochs", RuntimeWarning)

        # Save
        fif_path = os.path.join(output_dir, f'{subject_id}-epo.fif')
        npy_path = os.path.join(output_dir, f'{subject_id}_epochs.npy')
        epochs.save(fif_path, overwrite=True, verbose=False)
        np.save(npy_path, epochs.get_data())

        ch_path = os.path.join(output_dir, 'channel_names.txt')
        if not os.path.exists(ch_path):
            with open(ch_path, 'w') as f:
                f.write('\n'.join(epochs.ch_names))

    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)

    return result

Run batch over all subjects

In [7]:

print('=' * 70)
print('LEMON DATASET - FULL BATCH PREPROCESSING')
print('=' * 70)
print(f'Subjects to process: {len(subject_dirs)}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Target sampling rate: {TARGET_SFREQ} Hz')
print(f"GPU acceleration: {'YES (CuPy)' if MNE_CUDA else 'NO'}")
print('=' * 70 + '\n')

results = []
start_time = time.time()
active_attempts = 0
success_count = 0
failed_count = 0
missing_count = 0
skipped_count = 0

for idx, subject_dir in enumerate(subject_dirs):
    subject_id = os.path.basename(subject_dir)
    vhdr_file = os.path.join(subject_dir, 'RSEEG', f'{subject_id}.vhdr')

    output_npy = os.path.join(OUTPUT_DIR, f'{subject_id}_epochs.npy')
    if os.path.exists(output_npy):
        print(f'[{idx+1:3d}/{len(subject_dirs)}] {subject_id} - SKIPPED (already done)')
        skipped_count += 1
        results.append({'subject': subject_id, 'status': 'skipped_existing'})
        wandb_subject_rows.append({
            'subject': subject_id,
            'status': 'skipped_existing',
            'elapsed_s': 0.0,
            'n_bad_channels': 0,
            'n_rejected_ics': 0,
            'n_epochs_before': 0,
            'n_epochs_after': 0,
            'bad_segments_s': 0.0,
            'success_rate_running': success_count / max(1, active_attempts),
        })
        if wandb_run is not None:
            wandb_run.log({
                'preprocessing/subject_index': idx + 1,
                'preprocessing/subject_total': len(subject_dirs),
                'preprocessing/skipped_existing_count': skipped_count,
                'preprocessing/success_count': success_count,
                'preprocessing/failed_count': failed_count,
                'preprocessing/missing_count': missing_count,
                'preprocessing/active_attempts': active_attempts,
                'preprocessing/current_success_rate': success_count / max(1, active_attempts),
            }, step=idx + 1)
        continue

    if not os.path.exists(vhdr_file):
        print(f'[{idx+1:3d}/{len(subject_dirs)}] {subject_id} - SKIPPED (no .vhdr file)')
        missing_count += 1
        active_attempts += 1
        result = {'subject': subject_id, 'status': 'missing_file'}
        results.append(result)
        wandb_subject_rows.append({
            'subject': subject_id,
            'status': 'missing_file',
            'elapsed_s': 0.0,
            'n_bad_channels': 0,
            'n_rejected_ics': 0,
            'n_epochs_before': 0,
            'n_epochs_after': 0,
            'bad_segments_s': 0.0,
            'success_rate_running': success_count / max(1, active_attempts),
        })
        if wandb_run is not None:
            wandb_run.log({
                'preprocessing/subject_index': idx + 1,
                'preprocessing/subject_total': len(subject_dirs),
                'preprocessing/missing_count': missing_count,
                'preprocessing/success_count': success_count,
                'preprocessing/failed_count': failed_count,
                'preprocessing/skipped_existing_count': skipped_count,
                'preprocessing/active_attempts': active_attempts,
                'preprocessing/current_success_rate': success_count / max(1, active_attempts),
            }, step=idx + 1)
        continue

    t0 = time.time()
    print(f'[{idx+1:3d}/{len(subject_dirs)}] {subject_id} - Processing...', end=' ', flush=True)

    result = preprocess_lemon_subject(
        vhdr_file=vhdr_file,
        output_dir=OUTPUT_DIR,
        target_sfreq=TARGET_SFREQ,
        line_noise_freq=LINE_NOISE_FREQ,
        highpass_freq=HIGHPASS_FREQ,
        lowpass_freq=LOWPASS_FREQ,
        ic_threshold=IC_REJECTION_THRESHOLD,
        epoch_duration=EPOCH_DURATION,
        epoch_overlap_frac=EPOCH_OVERLAP,
        amplitude_reject_uv=AMPLITUDE_REJECT_UV,
        random_state=RANDOM_STATE,
        use_cuda=MNE_CUDA
    )

    elapsed = time.time() - t0
    result['elapsed_s'] = elapsed
    active_attempts += 1
    results.append(result)

    if result['status'] == 'success':
        success_count += 1
        shape = result['epoch_shape']
        print(f"Done {elapsed:.1f}s | {shape[0]} epochs | {result['n_bad_channels']} bad ch | {result['n_rejected_ics']} ICs rejected")
    else:
        failed_count += 1
        print(f"FAILED: {result.get('error', 'unknown')}")

    current_success_rate = success_count / max(1, active_attempts)
    wandb_subject_rows.append({
        'subject': subject_id,
        'status': result['status'],
        'elapsed_s': round(elapsed, 3),
        'n_bad_channels': result.get('n_bad_channels', 0),
        'n_rejected_ics': result.get('n_rejected_ics', 0),
        'n_epochs_before': result.get('n_epochs_before', 0),
        'n_epochs_after': result.get('n_epochs_after', 0),
        'bad_segments_s': result.get('bad_segments_s', 0.0),
        'success_rate_running': current_success_rate,
    })

    if wandb_run is not None:
        wandb_run.log({
            'preprocessing/subject_index': idx + 1,
            'preprocessing/subject_total': len(subject_dirs),
            'preprocessing/elapsed_s': elapsed,
            'preprocessing/bad_channels': result.get('n_bad_channels', 0),
            'preprocessing/rejected_ics': result.get('n_rejected_ics', 0),
            'preprocessing/epochs_before': result.get('n_epochs_before', 0),
            'preprocessing/epochs_after': result.get('n_epochs_after', 0),
            'preprocessing/bad_segments_s': result.get('bad_segments_s', 0.0),
            'preprocessing/success_count': success_count,
            'preprocessing/failed_count': failed_count,
            'preprocessing/missing_count': missing_count,
            'preprocessing/skipped_existing_count': skipped_count,
            'preprocessing/active_attempts': active_attempts,
            'preprocessing/current_success_rate': current_success_rate,
        }, step=idx + 1)

    if (idx + 1) % 10 == 0:
        print(f"W&B progress checkpoint: {idx+1}/{len(subject_dirs)} subjects logged")

total_time = time.time() - start_time
print('\n' + '=' * 70)
print('BATCH COMPLETE')
print('=' * 70)
n_success = sum(1 for r in results if r.get('status') == 'success')
n_failed = sum(1 for r in results if r.get('status') == 'failed')
print(f'Total time: {total_time/60:.1f} minutes')
print(f'Successful: {n_success}/{len(results)}')
print(f'Failed: {n_failed}/{len(results)}')
print(f'Skipped existing: {skipped_count}')
print(f'Missing files: {missing_count}')
print(f'Active success rate: {success_count}/{max(1, active_attempts)} = {success_count / max(1, active_attempts):.1%}')

LEMON DATASET - FULL BATCH PREPROCESSING
Subjects to process: 220
Output directory: G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\data\\LEMON_preprocessed
Target sampling rate: 250 Hz
GPU acceleration: YES (CuPy)

[  1/220] sub-010002 - SKIPPED (already done)
[  2/220] sub-010003 - SKIPPED (already done)
[  3/220] sub-010004 - SKIPPED (already done)
[  4/220] sub-010005 - SKIPPED (already done)
[  5/220] sub-010006 - SKIPPED (already done)
[  6/220] sub-010007 - SKIPPED (already done)
[  7/220] sub-010010 - SKIPPED (already done)
[  8/220] sub-010012 - SKIPPED (already done)
[  9/220] sub-010015 - SKIPPED (already done)
[ 10/220] sub-010016 - SKIPPED (already done)
[ 11/220] sub-010017 - SKIPPED (already done)
[ 12/220] sub-010019 - SKIPPED (already done)
[ 13/220] sub-010020 - Processing... 

FAILED: [Errno 2] No such file or directory: 'G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\EEG_MPILMBB_LEMON\\EEG_Raw_BIDS_ID\\sub-010020\\RSEEG\\Untitled.vmrk'
[ 14/220] sub-010021 - SKIPPED (already done)
[ 15/220] sub-010022 - SKIPPED (already done)
[ 16/220] sub-010023 - SKIPPED (already done)
[ 17/220] sub-010024 - SKIPPED (already done)
[ 18/220] sub-010026 - SKIPPED (already done)
[ 19/220] sub-010027 - SKIPPED (already done)
[ 20/220] sub-010028 - SKIPPED (already done)
[ 21/220] sub-010029 - SKIPPED (already done)
[ 22/220] sub-010030 - SKIPPED (already done)
[ 23/220] sub-010031 - SKIPPED (already done)
[ 24/220] sub-010032 - SKIPPED (already done)
[ 25/220] sub-010033 - SKIPPED (already done)
[ 26/220] sub-010034 - SKIPPED (already done)
[ 27/220] sub-010035 - SKIPPED (already done)
[ 28/220] sub-010036 - SKIPPED (already done)
[ 29/220] sub-010037 - SKIPPED (already done)
[ 30/220] sub-010038 - SKIPPED (already done)
[ 31/220] sub-010039 - SKIPPED 

C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:100: RuntimeWarning: High IC rejection fraction: 30.0%
  warnings.warn(f"High IC rejection fraction: {ic_frac:.1%}", RuntimeWarning)
C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:106: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])


Done 296.5s | 539 epochs | 1 bad ch | 18 ICs rejected
[180/220] sub-010278 - Processing... 

C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:100: RuntimeWarning: High IC rejection fraction: 36.1%
  warnings.warn(f"High IC rejection fraction: {ic_frac:.1%}", RuntimeWarning)


...

Done 690.1s | 533 epochs | 1 bad ch | 16 ICs rejected
[218/220] sub-010318 - Processing... 

C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:106: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])


Done 322.3s | 538 epochs | 1 bad ch | 10 ICs rejected
[219/220] sub-010319 - Processing... 

C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:100: RuntimeWarning: High IC rejection fraction: 27.1%
  warnings.warn(f"High IC rejection fraction: {ic_frac:.1%}", RuntimeWarning)
C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:106: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])
C:\Users\milon\AppData\Local\Temp\ipykernel_9940\1095459898.py:106: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])


Done 187.6s | 506 epochs | 2 bad ch | 16 ICs rejected
[220/220] sub-010321 - Processing... Done 2173.4s | 493 epochs | 0 bad ch | 9 ICs rejected
W&B progress checkpoint: 220/220 subjects logged

BATCH COMPLETE
Total time: 517.0 minutes
Successful: 40/220
Failed: 4/220
Skipped existing: 171
Missing files: 5
Active success rate: 40/49 = 81.6%


Summary, CSV export and visualizations

In [ ]:

successful = [r for r in results if r.get('status') == 'success']

#building a  DataFrame across all subjects first so we can always export the full run log
df_all = pd.DataFrame(results)

# Ensure the columns we use for plotting exist even when some subjects failed early.
for col, default in {
    'n_bad_channels': 0,
    'n_rejected_ics': 0,
    'reject_ics_fraction': np.nan,
    'n_epochs_before': 0,
    'n_epochs_after': 0,
    'epoch_keep_ratio': np.nan,
    'bad_segments_s': 0.0,
    'elapsed_s': np.nan,
}.items():
    if col not in df_all.columns:
        df_all[col] = default
    else:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# Normalise the bad-channel list column for downstream plotting.
if 'bad_channel_names' not in df_all.columns:
    df_all['bad_channel_names'] = [[] for _ in range(len(df_all))]
else:
    df_all['bad_channel_names'] = df_all['bad_channel_names'].apply(lambda x: x if isinstance(x, list) else [])

df_all['success_binary'] = (df_all['status'] == 'success').astype(int)

detailed_metrics_path = os.path.join(OUTPUT_DIR, 'preprocessing_metrics_per_subject.csv')
df_all.to_csv(detailed_metrics_path, index=False)
print(f'Per-subject metrics saved: {detailed_metrics_path}')


bad_channel_rows = []
for _, row in df_all.iterrows():
    for ch in row['bad_channel_names']:
        bad_channel_rows.append({'subject': row.get('subject', ''), 'channel': ch})

if bad_channel_rows:
    df_bad_channels = pd.DataFrame(bad_channel_rows)
    bad_channel_freq = (
        df_bad_channels.groupby('channel')
        .size()
        .sort_values(ascending=False)
        .reset_index(name='n_subjects')
    )
else:
    df_bad_channels = pd.DataFrame(columns=['subject', 'channel'])
    bad_channel_freq = pd.DataFrame(columns=['channel', 'n_subjects'])

bad_channel_freq_path = os.path.join(OUTPUT_DIR, 'bad_channel_frequency.csv')
bad_channel_freq.to_csv(bad_channel_freq_path, index=False)
print(f'Bad-channel frequency saved: {bad_channel_freq_path}')

# Create output folder for figures.
figdir = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(figdir, exist_ok=True)

# Styling for publication-friendly plots.
sns.set_theme(style='whitegrid', context='talk', palette='deep')
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 320,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Order subjects for readable per-subject figures.
subject_order = df_all['subject'].astype(str).tolist() if 'subject' in df_all.columns else list(range(len(df_all)))
status_palette = {
    'success': '#2a9d8f',
    'failed': '#e76f51',
    'missing_file': '#f4a261',
    'skipped_existing': '#457b9d',
}

# 1) Final outcome per subject.
if not df_all.empty and 'subject' in df_all.columns:
    plt.figure(figsize=(max(12, 0.42 * len(df_all)), 5))
    ax = sns.barplot(
        data=df_all,
        x='subject',
        y='success_binary',
        order=subject_order,
        hue='status',
        dodge=False,
        palette=status_palette,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title('Per-subject preprocessing outcome')
    ax.set_xlabel('Subject')
    ax.set_ylabel('Success indicator')
    ax.set_ylim(0, 1.1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Failed / not processed', 'Success'])
    plt.xticks(rotation=90)
    plt.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'subject_success_outcome.png'))
    plt.savefig(os.path.join(figdir, 'subject_success_outcome.svg'))
    plt.show()

# 2) Bad channels per subject.
if not df_all.empty:
    plt.figure(figsize=(max(12, 0.42 * len(df_all)), 5))
    ax = sns.barplot(
        data=df_all,
        x='subject',
        y='n_bad_channels',
        order=subject_order,
        color='#264653',
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title('Bad channels detected per subject')
    ax.set_xlabel('Subject')
    ax.set_ylabel('Number of bad channels')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'subject_bad_channels.png'))
    plt.savefig(os.path.join(figdir, 'subject_bad_channels.svg'))
    plt.show()

# 3) ICA rejection fraction per subject.
if not df_all.empty:
    plt.figure(figsize=(max(12, 0.42 * len(df_all)), 5))
    ax = sns.barplot(
        data=df_all,
        x='subject',
        y='reject_ics_fraction',
        order=subject_order,
        hue='status',
        dodge=False,
        palette=status_palette,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title('ICA rejection fraction per subject')
    ax.set_xlabel('Subject')
    ax.set_ylabel('Rejected ICA fraction')
    ax.set_ylim(0, max(0.05, np.nanmax(df_all['reject_ics_fraction'].fillna(0)) * 1.15))
    plt.xticks(rotation=90)
    plt.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'subject_ic_rejection_fraction.png'))
    plt.savefig(os.path.join(figdir, 'subject_ic_rejection_fraction.svg'))
    plt.show()

# 4) Per-subject epoch keep ratio.
if not df_all.empty:
    plt.figure(figsize=(max(12, 0.42 * len(df_all)), 5))
    ax = sns.barplot(
        data=df_all,
        x='subject',
        y='epoch_keep_ratio',
        order=subject_order,
        hue='status',
        dodge=False,
        palette=status_palette,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title('Epoch retention per subject')
    ax.set_xlabel('Subject')
    ax.set_ylabel('Epoch keep ratio')
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=90)
    plt.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'subject_epoch_keep_ratio.png'))
    plt.savefig(os.path.join(figdir, 'subject_epoch_keep_ratio.svg'))
    plt.show()

# 5) Bad-channel frequency across the cohort.
if not bad_channel_freq.empty:
    top_n = min(20, len(bad_channel_freq))
    plt.figure(figsize=(10, max(5, 0.35 * top_n)))
    plot_df = bad_channel_freq.head(top_n).sort_values('n_subjects', ascending=True)
    ax = sns.barplot(data=plot_df, y='channel', x='n_subjects', color='#1d3557')
    ax.set_title('Most frequently detected bad channels')
    ax.set_xlabel('Number of subjects flagged')
    ax.set_ylabel('Channel')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'bad_channel_frequency_top20.png'))
    plt.savefig(os.path.join(figdir, 'bad_channel_frequency_top20.svg'))
    plt.show()

# 6) Heatmap of bad channels by subject.
if not df_bad_channels.empty:
    heatmap_df = (
        df_bad_channels.assign(value=1)
        .pivot_table(index='subject', columns='channel', values='value', aggfunc='max', fill_value=0)
    )
    heatmap_df = heatmap_df.loc[subject_order]
    plt.figure(figsize=(max(12, 0.35 * heatmap_df.shape[1]), max(6, 0.25 * heatmap_df.shape[0])))
    ax = sns.heatmap(
        heatmap_df,
        cmap='mako',
        cbar_kws={'label': 'Bad channel detected'},
        linewidths=0.15,
        linecolor='white',
    )
    ax.set_title('Bad-channel incidence heatmap')
    ax.set_xlabel('Channel')
    ax.set_ylabel('Subject')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'bad_channel_heatmap.png'))
    plt.savefig(os.path.join(figdir, 'bad_channel_heatmap.svg'))
    plt.show()

# 7) Rejection statistics and runtime diagnostics.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df_all['reject_ics_fraction'].dropna(), bins=20, kde=True, ax=axes[0, 0], color='#e63946')
axes[0, 0].set_title('Distribution of ICA rejection fraction')
axes[0, 0].set_xlabel('Rejected ICA fraction')

sns.histplot(df_all['n_bad_channels'].dropna(), bins=range(0, int(df_all['n_bad_channels'].max() or 0) + 3), ax=axes[0, 1], color='#264653')
axes[0, 1].set_title('Distribution of bad-channel counts')
axes[0, 1].set_xlabel('Bad channels per subject')

sns.scatterplot(data=df_all, x='n_bad_channels', y='n_rejected_ics', hue='status', palette=status_palette, ax=axes[1, 0])
axes[1, 0].set_title('Bad channels vs rejected ICA components')
axes[1, 0].set_xlabel('Bad channels')
axes[1, 0].set_ylabel('Rejected ICs')

sns.boxplot(data=df_all, x='status', y='epoch_keep_ratio', palette=status_palette, ax=axes[1, 1])
axes[1, 1].set_title('Epoch keep ratio by status')
axes[1, 1].set_xlabel('Status')
axes[1, 1].set_ylabel('Keep ratio')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(figdir, 'summary_diagnostics_grid.png'))
plt.savefig(os.path.join(figdir, 'summary_diagnostics_grid.svg'))
plt.show()

# Optional runtime summary.
if 'elapsed_s' in df_all.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(df_all['elapsed_s'].dropna() / 60.0, bins=20, color='#6c757d')
    plt.title('Processing time per subject')
    plt.xlabel('Minutes')
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, 'processing_time_minutes_hist.png'))
    plt.savefig(os.path.join(figdir, 'processing_time_minutes_hist.svg'))
    plt.show()

# Print concise text summary.
if successful:
    df = pd.DataFrame(successful)

    print('=' * 60)
    print(f'PREPROCESSING SUMMARY ({len(successful)} subjects)')
    print('=' * 60)
    print(f"Successful: {len(successful)}")
    print(f"Failed: {failed_count}")
    print(f"Missing files: {missing_count}")
    print(f"Skipped existing: {skipped_count}")
    print(f"Mean bad channels: {df['n_bad_channels'].mean():.2f}")
    print(f"Mean rejected ICs: {df['n_rejected_ics'].mean():.2f}")
    print(f"Mean epoch keep ratio: {df['epoch_keep_ratio'].mean():.2f}")

    if wandb_run is not None:
        wandb_table = wandb.Table(dataframe=pd.DataFrame(wandb_subject_rows))
        wandb_run.log({
            'preprocessing/final_success_rate': success_count / max(1, active_attempts),
            'preprocessing/total_subjects': len(subject_dirs),
            'preprocessing/successful_subjects': success_count,
            'preprocessing/failed_subjects': failed_count,
            'preprocessing/missing_subjects': missing_count,
            'preprocessing/skipped_existing_subjects': skipped_count,
            'preprocessing/active_attempts': active_attempts,
            'preprocessing/summary_table': wandb_table,
        })
        wandb_run.summary['final_success_rate'] = success_count / max(1, active_attempts)
        wandb_run.summary['successful_subjects'] = success_count
        wandb_run.summary['failed_subjects'] = failed_count
        wandb_run.summary['missing_subjects'] = missing_count
        wandb_run.summary['skipped_existing_subjects'] = skipped_count
        wandb_run.summary['active_attempts'] = active_attempts
        wandb_run.summary['summary_csv'] = detailed_metrics_path
        wandb_run.finish()
else:
    print('No successful results to summarize.')
    if wandb_run is not None:
        wandb_run.summary['final_success_rate'] = 0.0
        wandb_run.summary['successful_subjects'] = 0
        wandb_run.summary['failed_subjects'] = failed_count
        wandb_run.summary['missing_subjects'] = missing_count
        wandb_run.summary['skipped_existing_subjects'] = skipped_count
        wandb_run.summary['active_attempts'] = active_attempts
        wandb_run.finish()

print('Figures saved to:', figdir)


## Documentation Files

The full written documentation for this notebook is available here:

- [Markdown version](docs/LEMON_FULL_BATCH_PREPROCESSING_DOCUMENTATION.md)
- [Word version](docs/LEMON_FULL_BATCH_PREPROCESSING_DOCUMENTATION.docx)

A short index is also available in [docs/README.md](docs/README.md).

## Preprocessing Pipeline Diagram (Mermaid Flowchart)

```mermaid
graph TD
    A["📥 Raw BrainVision EEG<br/>(128ch, variable sfreq)"] --> B["1️⃣ Channel Setup<br/>(montage, reference)"]
    B --> C["2️⃣ Filter & Resample<br/>(notch, bandpass, 250Hz)"]
    C --> D["3️⃣ Bad Channel Detection<br/>(PyPREP: deviation, NaN, correlation)"]
    D --> E["📊 Store: bad_channels<br/>n_bad_channels"]
    E --> F["4️⃣ Drop Bad Channels<br/>(temporary, for ICA)"]
    F --> G["5️⃣ Re-reference<br/>(average reference)"]
    G --> H["6️⃣ ICA Decomposition<br/>(infomax, extended)"]
    H --> I["7️⃣ ICLabel Classification<br/>(eye, muscle, line noise...)"]
    I --> J["8️⃣ Threshold-based IC Rejection<br/>(P ≥ 0.80 → exclude)"]
    J --> E1["📊 Store: n_rejected_ics<br/>reject_ics_fraction"]
    E1 --> K{"⚠️ QC1: IC Rejection<br/>Fraction > 0.25?"}
    K -->|Yes| K1["🚩 Flag: High artifact<br/>contamination"]
    K -->|No| L["9️⃣ Apply ICA<br/>(reconstruct w/o rejected)"]
    K1 --> L
    L --> M["🔟 Restore & Interpolate<br/>Bad Channels"]
    M --> N["1️⃣1️⃣ Bad Segment Annotation<br/>(sliding window, 250µV)"]
    N --> O["📊 Store: bad_segments_s"]
    O --> P["1️⃣2️⃣ Create Epochs<br/>(4s, 50% overlap)"]
    P --> Q["📊 Store: n_epochs_before"]
    Q --> R["1️⃣3️⃣ Epoch Rejection<br/>(amplitude > 250µV)"]
    R --> S["📊 Store: n_epochs_after<br/>epoch_keep_ratio"]
    S --> T{"⚠️ QC2: Epoch Keep<br/>Ratio < 0.50?"}
    T -->|Yes| T1["🚩 Flag: Large epoch loss"]
    T -->|No| U["1️⃣4️⃣ Save Outputs<br/>(.fif, .npy, channel_names.txt)"]
    T1 --> U
    U --> V["✅ COMPLETE<br/>(ready for ML/analysis)"]
    
    style A fill:#e1f5ff
    style V fill:#c8e6c9
    style K fill:#fff9c4
    style T fill:#fff9c4
    style K1 fill:#ffccbc
    style T1 fill:#ffccbc
    style E fill:#f3e5f5
    style E1 fill:#f3e5f5
    style O fill:#f3e5f5
    style Q fill:#f3e5f5
    style S fill:#f3e5f5
```

This flowchart shows:
- **Light blue** = inputs
- **Light green** = successful completion
- **Light purple** = stored metrics/results
- **Yellow** = quality control checkpoints
- **Orange** = warning flags

---

Full detailed diagram document: [PREPROCESSING_PIPELINE_DIAGRAM.md](docs/PREPROCESSING_PIPELINE_DIAGRAM.md)
